## 🧬 Phylogenetic Tree Construction with FastTree

This notebook generates rooted phylogenetic trees for aligned sequence files using **FastTree**.

### 📁 Input Folder:
- `final_forward/` — contains all trimmed, aligned FASTA files with outgroups prepended.
- Each file is named as: `gene_direction.fasta` (e.g., `alaS_forward.fasta`, `iolB_reverse.fasta`).

### 🌲 Output Folder:
- `trees_final/` — will be created automatically.
- Stores each corresponding tree in Newick format (`_tree.nwk` suffix), for example: `alaS_forward_tree.nwk`.

### 🚀 Workflow Steps:
1. Loop through all `.fasta` files in `final_fastas/`.
2. Run **FastTree** on each file using nucleotide mode (`-nt`).
3. Save the resulting tree to `trees_final/`.

> FastTree uses the Jukes-Cantor model and produces trees quickly even with large datasets.

In [1]:
import os
import subprocess
from Bio import SeqIO, Phylo
from io import StringIO
from collections import defaultdict, Counter

## 🧹 Enforcing Unique Headers (Preserving Outgroup)

**Why?**  
FastTree requires all sequence headers to be unique. However, in our datasets, the first sequence is always the **outgroup**, and we want to preserve its original header.

**Solution**  
We process each FASTA file to:
- Leave the **first sequence (outgroup)** untouched.
- Assign unique names (`seq1`, `seq2`, ...) to all remaining sequences.
- Preserve the original header in the description.

In [79]:
def enforce_clean_and_unique_headers(fasta_folder):
    for filename in os.listdir(fasta_folder):
        if not filename.endswith(".fasta"):
            continue

        input_path = os.path.join(fasta_folder, filename)
        output_lines = []
        sequence_index = 0
        seen = defaultdict(int)

        with open(input_path, "r") as f:
            lines = f.readlines()

        i = 0
        while i < len(lines):
            if lines[i].startswith(">"):
                raw_header = lines[i].strip()[1:]
                sequence_lines = []
                i += 1
                while i < len(lines) and not lines[i].startswith(">"):
                    sequence_lines.append(lines[i])
                    i += 1

                if sequence_index == 0:
                    # OUTGROUP: just take first part before ":" (drop extra description)
                    cleaned_header = raw_header.split(":")[0]
                else:
                    # REGULAR: keep first part before ":" and preserve any trailing "-envX"
                    id_part = raw_header.split(":")[0]
                    env_part = raw_header.split("-")[-1] if "-env" in raw_header else ""
                    base_id = f"{id_part}-{env_part}" if env_part else id_part

                    # Ensure uniqueness
                    seen[base_id] += 1
                    cleaned_header = f"{base_id}_v{seen[base_id]}" if seen[base_id] > 1 else base_id

                output_lines.append(f">{cleaned_header}\n")
                output_lines.extend(sequence_lines)
                sequence_index += 1
            else:
                i += 1

        with open(input_path, "w") as f:
            f.writelines(output_lines)

        print(f"🔧 Fixed headers (clean + unique) in: {filename}")


In [80]:
# Run this
enforce_clean_and_unique_headers("final_fastas")

🔧 Fixed headers (clean + unique) in: acsA_forward.fasta
🔧 Fixed headers (clean + unique) in: acsA_reverse.fasta
🔧 Fixed headers (clean + unique) in: acuA_forward.fasta
🔧 Fixed headers (clean + unique) in: acuA_reverse.fasta
🔧 Fixed headers (clean + unique) in: alaS_forward.fasta
🔧 Fixed headers (clean + unique) in: alaS_reverse.fasta
🔧 Fixed headers (clean + unique) in: albG_forward.fasta
🔧 Fixed headers (clean + unique) in: albG_reverse.fasta
🔧 Fixed headers (clean + unique) in: alkH_forward.fasta
🔧 Fixed headers (clean + unique) in: alkH_reverse.fasta
🔧 Fixed headers (clean + unique) in: ecfA1_forward.fasta
🔧 Fixed headers (clean + unique) in: ecfA1_reverse.fasta
🔧 Fixed headers (clean + unique) in: iolB_forward.fasta
🔧 Fixed headers (clean + unique) in: iolB_reverse.fasta
🔧 Fixed headers (clean + unique) in: opuAB_forward.fasta
🔧 Fixed headers (clean + unique) in: opuAB_reverse.fasta
🔧 Fixed headers (clean + unique) in: rmlD_forward.fasta
🔧 Fixed headers (clean + unique) in: rmlD_re

Now trim the outgroup to the modal sequence length (which should be the length of all sequences)

In [2]:
def trim_outgroup_to_modal_length(fasta_dir):
    for filename in sorted(os.listdir(fasta_dir)):
        if not filename.endswith(".fasta"):
            continue

        path = os.path.join(fasta_dir, filename)
        with open(path, "r") as f:
            lines = f.readlines()

        # Split into sequences
        sequences = []
        current_header = None
        current_seq = []

        for line in lines:
            if line.startswith(">"):
                if current_header is not None:
                    sequences.append((current_header, ''.join(current_seq)))
                current_header = line.strip()
                current_seq = []
            else:
                current_seq.append(line.strip())
        if current_header:
            sequences.append((current_header, ''.join(current_seq)))

        # Compute modal length (excluding the outgroup, which is first)
        non_outgroup_lengths = [len(seq) for _, seq in sequences[1:]]
        mode = Counter(non_outgroup_lengths).most_common(1)[0][0]

        # Trim the outgroup (first sequence)
        outgroup_header, outgroup_seq = sequences[0]
        trimmed_outgroup_seq = outgroup_seq[:mode]
        sequences[0] = (outgroup_header, trimmed_outgroup_seq)

        # Write back
        with open(path, "w") as f:
            for header, seq in sequences:
                f.write(f"{header}\n")
                for i in range(0, len(seq), 60):
                    f.write(seq[i:i+60] + "\n")

        print(f"✂️ Trimmed outgroup in {filename} to length {mode}")


✂️ Trimmed outgroup in acsA_forward.fasta to length 301
✂️ Trimmed outgroup in acsA_reverse.fasta to length 301
✂️ Trimmed outgroup in acuA_forward.fasta to length 301
✂️ Trimmed outgroup in acuA_reverse.fasta to length 301
✂️ Trimmed outgroup in alaS_forward.fasta to length 301
✂️ Trimmed outgroup in alaS_reverse.fasta to length 301
✂️ Trimmed outgroup in albG_forward.fasta to length 301
✂️ Trimmed outgroup in albG_reverse.fasta to length 301
✂️ Trimmed outgroup in alkH_forward.fasta to length 301
✂️ Trimmed outgroup in alkH_reverse.fasta to length 301
✂️ Trimmed outgroup in ecfA1_forward.fasta to length 301
✂️ Trimmed outgroup in ecfA1_reverse.fasta to length 301
✂️ Trimmed outgroup in iolB_forward.fasta to length 301
✂️ Trimmed outgroup in iolB_reverse.fasta to length 301
✂️ Trimmed outgroup in opuAB_forward.fasta to length 301
✂️ Trimmed outgroup in opuAB_reverse.fasta to length 301
✂️ Trimmed outgroup in rmlD_forward.fasta to length 301
✂️ Trimmed outgroup in rmlD_reverse.fasta to

In [ ]:
# Run it on your FASTA directory
trim_outgroup_to_modal_length("final_fastas")

Now make trees

In [73]:
def make_trees_batch(final_folder="tree_rdy_fastas", tree_folder="trees_final", threshold=100_000):
    os.makedirs(tree_folder, exist_ok=True)
    fasttree_exe = r"C:\ecosim\bin\fasttree.exe"  # Windows-style path

    for file in os.listdir(final_folder):
        if file.endswith(".fasta"):
            input_path = os.path.join(final_folder, file)
            output_path = os.path.join(tree_folder, file.replace(".fasta", "_tree.nwk"))

            # Count sequences
            num_seqs = sum(1 for _ in SeqIO.parse(input_path, "fasta"))
            print(f"🌲 Making tree for {file} ({num_seqs} sequences)...")

            # Choose FastTree options
            if num_seqs > threshold:
                print("⚡ Large file detected. Using optimized FastTree options.")
                cmd = [
                    fasttree_exe, "-nt", "-fastest", "-nosupport", "-gtr", input_path
                ]
            else:
                cmd = [
                    fasttree_exe, "-nt", input_path
                ]

            try:
                result = subprocess.run(cmd, capture_output=True, text=True)
                if result.returncode == 0:
                    with open(output_path, "w") as f:
                        f.write(result.stdout)
                    print(f"✅ Tree written to {output_path}")
                else:
                    print(f"❌ FastTree error for {file}:\n{result.stderr.strip()}")
            except Exception as e:
                print(f"💥 Exception while processing {file}: {str(e)}")



In [ ]:
# Just call it directly since we assume arguments
make_trees_batch("final_fastas")

🌲 Making tree for acsA_forward.fasta (347960 sequences)...
⚡ Large file detected. Using optimized FastTree options.
❌ FastTree error for acsA_forward.fasta:
FastTree Version 2.1.11 Double precision (No SSE3), OpenMP (8 threads)
Alignment: final_fastas\acsA_forward.fasta
Nucleotide distances: Jukes-Cantor Joins: balanced Support: none
Search: Fastest+2nd +NNI +SPR (2 rounds range 10) +ML-NNI opt-each=1
TopHits: 1.00*sqrtN close=default refresh=0.50
ML Model: Generalized Time-Reversible, CAT approximation with 20 rate categories
Wrong number of characters for AmpSeq-MD2-env1: expected 324 but have 301 instead.
This sequence may be truncated, or another sequence may be too long.
🌲 Making tree for acsA_reverse.fasta (348201 sequences)...
⚡ Large file detected. Using optimized FastTree options.
❌ FastTree error for acsA_reverse.fasta:
FastTree Version 2.1.11 Double precision (No SSE3), OpenMP (8 threads)
Alignment: final_fastas\acsA_reverse.fasta
Nucleotide distances: Jukes-Cantor Joins: ba

## 🧬 Constructing Phylogenetic Trees with FastTree extra notes

This step creates rooted phylogenetic trees from our final aligned FASTA files using **FastTree**.

- Trees are generated for each gene-direction pair (e.g., `acsA_forward.fasta`).
- If a file has over **100,000 sequences**, optimized FastTree options are used:
  - `-fastest` skips slow heuristics
  - `-nosupport` disables support value calculations
  - `-gtr` uses a more accurate evolutionary model

All trees are saved in the `trees_final/` directory.

## 🌳 Rerooting Trees with Outgroups (Non-Destructive)

This step takes each tree from the `trees_final` directory and reroots it using the first sequence from its matching FASTA file as the outgroup. 

### 🔁 What’s New
- Rerooted trees are **saved to a new directory**: `rerooted_trees`
- The original trees remain untouched for safety.

### 🧠 Why This Is Important
Rerooting helps orient each gene tree based on a known outgroup, giving a clearer evolutionary context.

### 🧪 Expected Input/Output
- **Input trees**: `trees_final/gene_forward_tree.nwk`
- **Input sequences**: `final_fastas/gene_forward.fasta`
- **Output trees**: `rerooted_trees/gene_forward_tree.nwk`

In [75]:
def sanitize_header(header):
    """Return a safe identifier for matching in the tree."""
    return header.replace(">", "").replace(" ", "_").split()[0]

def reroot_tree(tree_file, fasta_file, output_file):
    # Get the first (outgroup) sequence ID
    with open(fasta_file) as handle:
        outgroup_id_raw = next(SeqIO.parse(handle, "fasta")).id
        outgroup_id = sanitize_header(outgroup_id_raw)

    # Read and parse the Newick tree
    with open(tree_file) as f:
        tree = Phylo.read(StringIO(f.read()), "newick")

    # Try to match the outgroup
    outgroup = next(
        (clade for clade in tree.find_clades() if clade.name and outgroup_id in clade.name),
        None
    )

    if outgroup:
        tree.root_with_outgroup(outgroup)
        with open(output_file, "w") as f:
            Phylo.write(tree, f, "newick")
        print(f"✅ Rerooted: {os.path.basename(output_file)}")
    else:
        print(f"❌ Outgroup '{outgroup_id}' not found in {os.path.basename(tree_file)}")

def reroot_all_trees(tree_dir, fasta_dir, output_dir="rerooted_trees"):
    os.makedirs(output_dir, exist_ok=True)

    for filename in os.listdir(tree_dir):
        if filename.endswith("_tree.nwk"):
            base = filename.replace("_tree.nwk", "")
            tree_path = os.path.join(tree_dir, filename)
            fasta_path = os.path.join(fasta_dir, f"{base}.fasta")
            output_path = os.path.join(output_dir, filename)

            if os.path.exists(fasta_path):
                reroot_tree(tree_path, fasta_path, output_path)
            else:
                print(f"⚠️ Missing FASTA file: {fasta_path}")



In [ ]:
# Run:
reroot_all_trees("trees_final", "final_fastas")

✅ Rerooted: ecfA1_forward_tree.nwk
✅ Rerooted: ecfA1_reverse_tree.nwk
✅ Rerooted: opuAB_forward_tree.nwk
✅ Rerooted: opuAB_reverse_tree.nwk


Now we can run ecosim on our rerooted trees and fasta files proceed to
`ecosim_run`